# Machine Learning Cycle - Image Classification
This notebook documents data acquisition, preprocessing, model training, optimization, testing metrics, prediction, and retraining trigger flow.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.append(str(project_root / 'src'))
print('Project root:', project_root)

## 1. Data Acquisition
Dataset is image-based (non-tabular) and stored in train/test class folders under archive (14).

In [ ]:
from preprocessing import resolve_default_paths

paths = resolve_default_paths(project_root)
print('Train dir:', paths.train_dir)
print('Test dir :', paths.test_dir)

## 2. Data Processing
Preprocessing steps:
- resize images to fixed size
- convert RGB pixels into normalized vectors
- derive interpretable features: brightness, blue_ratio, green_ratio, texture_strength

In [ ]:
from preprocessing import load_split, sample_story_features

x_train, y_train, classes = load_split(paths.train_dir, max_per_class=200)
x_test, y_test, _ = load_split(paths.test_dir, max_per_class=100)
story_x, story_y = sample_story_features(paths.test_dir, classes, samples_per_class=20)

print('Train shape:', x_train.shape)
print('Test shape :', x_test.shape)
print('Classes    :', classes)
print('Feature rows:', story_x.shape[0])

## 3. Model Creation and Optimization
Optimization choices used:
- `LogisticRegression` with regularization (default L2)
- `solver='saga'` to handle multiclass efficiently
- tuned `max_iter` for better convergence
- standardization before classification

In [ ]:
from model import train_model, TrainingConfig

result = train_model(
    project_root,
    TrainingConfig(max_train_per_class=500, max_test_per_class=250, max_iter=60)
)
result

## 4. Model Testing and Evaluation Metrics
Explicit metrics reported:
- Accuracy
- Precision
- Recall
- F1-score
Additional metric:
- Confusion Matrix

In [ ]:
metrics_path = project_root / 'results' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))

print('Accuracy:', metrics['accuracy'])
report_df = pd.DataFrame(metrics['classification_report']).transpose()
display(report_df[['precision', 'recall', 'f1-score']].head(8))
print('Confusion Matrix size:', len(metrics['confusion_matrix']), 'x', len(metrics['confusion_matrix'][0]))

### Interpretation of Results
The baseline model captures useful class patterns but still confuses visually similar classes.
Main improvements can come from stronger models (CNN or transfer learning), more iterations, and class-balanced tuning.

## 5. Prediction Function Demo

In [ ]:
from prediction import predict_image

sample_image = sorted((paths.test_dir / classes[0]).glob('*.jpg'))[0]
prediction_output = predict_image(project_root, sample_image)
prediction_output

## 6. Retraining Trigger
Retraining flow for grading:
1. Upload images through dashboard/API (`/upload-bulk`) and save into `data/uploads/`.
2. Preprocessing is applied through `src/preprocessing.py`.
3. Retrain via `POST /retrain` or `python src/retrain.py`, reusing the established pipeline and regenerating model + metrics artifacts.